# Mock Run — ETTh1, 1 Epoch per Model

Quick smoke-test: runs all 7 models on ETTh1 (pred_len=96) for **1 epoch** each.  
Results are saved to Google Drive.

## 1 — Google Drive Auth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 — Configuration

In [ ]:
# Folder in Google Drive where results will be saved
GDRIVE_RESULTS_DIR = '/content/drive/MyDrive/multimodality/mock_results'

# Mock settings
MOCK_EPOCHS  = 1
MOCK_HORIZON = 96
MOCK_DATASET = 'ETTh1'

## 3 — Setup

In [ ]:
import os

# Detect project root (works whether run_mock.ipynb is at repo root or one level up)
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'run_mock.ipynb')):
    PROJECT_ROOT = _cwd
elif os.path.isfile(os.path.join(_cwd, 'multimodality', 'run_mock.ipynb')):
    PROJECT_ROOT = os.path.join(_cwd, 'multimodality')
else:
    PROJECT_ROOT = _cwd

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch

if torch.cuda.is_available():
    device_info = f'CUDA — {torch.cuda.get_device_name(0)}'
elif torch.backends.mps.is_available():
    device_info = 'Apple MPS'
else:
    device_info = 'CPU'

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device_info}')

## 4 — Run Mock Experiments

7 models × ETTh1 × pred_len=96 × **1 epoch** each.

In [ ]:
import json, pathlib

registry_path = pathlib.Path('experiments/registry.json')
registry_path.parent.mkdir(parents=True, exist_ok=True)
with open(registry_path, 'w') as f:
    json.dump([], f)
print('Registry cleared.')

In [ ]:
from run_experiment import run

CONFIGS = [
    ('experiments/configs/01_dlinear_etth1.yaml',         'dlinear'),
    ('experiments/configs/02_patchtst_etth1.yaml',        'patchtst'),
    ('experiments/configs/03_bert_forecaster_etth1.yaml', 'bert_forecaster'),
    ('experiments/configs/04_late_fusion_etth1.yaml',     'late_fusion'),
    ('experiments/configs/05_gated_fusion_etth1.yaml',    'gated_fusion'),
    ('experiments/configs/06_film_fusion_etth1.yaml',     'film_fusion'),
    ('experiments/configs/07_ensemble_fusion_etth1.yaml', 'ensemble_fusion'),
]

mock_results = {}

for config_path, label in CONFIGS:
    exp_name = f'mock_{label}_{MOCK_DATASET.lower()}_pred{MOCK_HORIZON}'
    overrides = [
        f'name={exp_name}',
        f'model.pred_len={MOCK_HORIZON}',
        f'training.train_epochs={MOCK_EPOCHS}',
    ]
    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')
    try:
        metrics = run(config_path, overrides=overrides)
        mock_results[label] = metrics
    except SystemExit:
        print(f'Skipped {label} — GPU required but not available.')
        mock_results[label] = None
    except Exception as e:
        print(f'Error in {label}: {e}')
        mock_results[label] = None

print('\nAll mock experiments finished.')

## 5 — Save Results to Google Drive

In [ ]:
import shutil, pathlib
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dest = pathlib.Path(GDRIVE_RESULTS_DIR) / timestamp
dest.mkdir(parents=True, exist_ok=True)

shutil.copy('experiments/registry.json', dest / 'registry.json')

src_results = pathlib.Path('experiments/results')
if src_results.exists():
    shutil.copytree(src_results, dest / 'results', dirs_exist_ok=True)

print(f'Saved to: {dest}')

## 6 — Results Table

In [ ]:
from compare_results import load_registry, print_table

records = load_registry()
mock_records = [r for r in records if r['name'].startswith('mock_')]

if mock_records:
    print(f'\n=== Mock Run | {MOCK_DATASET} | pred_len={MOCK_HORIZON} | {MOCK_EPOCHS} epoch ===')
    print_table(mock_records, sort_by='mse')
else:
    print('No mock results found in registry.')